<a href="https://colab.research.google.com/github/laurindodumba/AULA-FUNDAMENTO-DE-INTELIGENCIA-ARTIFICIAL/blob/main/Aula_06_Chat_PDF_corrigido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Aula 06 — Chat com PDF

Pipeline RAG usando LangChain moderno (1.x) + HuggingFace (gratuito e local).

**Fluxo:**
1. Carregar PDF
2. Quebrar em chunks
3. Gerar embeddings (HuggingFace, gratuito)
4. Guardar no banco vetorial (Chroma)
5. Retriever busca os trechos relevantes
6. LLM responde com base no contexto (via LCEL)

In [15]:
# Instalação única — versões compatíveis
!pip install -q \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-core \
    chromadb \
    sentence-transformers \
    transformers \
    pypdf \
    huggingface_hub \
    accelerate

In [16]:
# Imports — todos de pacotes que existem no LangChain
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline

print("Imports OK!")

Imports OK!


In [18]:
# 1. Carregar PDF
# Faça upload do seu PDF no Colab antes de executar esta célula
arquivo = '/content/pdf_1772.pdf'

loader = PyPDFLoader(arquivo)
documents = loader.load()

print(f"Total de páginas carregadas: {len(documents)}")
print("\nPrimeira página (trecho):")
print(documents[0].page_content[:500])

Total de páginas carregadas: 5

Primeira página (trecho):
Gestão de Pessoas e Psicologia Organizacional e do Trabalho
Se você deseja impulsionar sua carreira no universo corporativo, se tornar um profissional altamente capacitado e desenvolver uma
visão estratégica sobre a gestão de pessoas, a nossa especialização em Gestão de Pessoas e Psicologia Organizacional e do
Trabalho é a oportunidade que você estava esperando. 
Com uma formação focada em resultados práticos e soluções inovadoras, este curso é ideal para quem quer se destacar no
mercado de trab


In [19]:
# 2. Quebrar texto em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=80
)
docs = text_splitter.split_documents(documents)

print(f"Total de chunks gerados: {len(docs)}")

Total de chunks gerados: 57


In [20]:
# 3. Embeddings + Banco vetorial + Retriever
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(docs, embeddings)

retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3}
)

print("Banco vetorial e retriever prontos!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Banco vetorial e retriever prontos!


In [22]:
# 4. LLM local

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200,
)

llm = HuggingFacePipeline(pipeline=pipe)

print("Modelo LLM carregado!")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

Modelo LLM carregado!


In [23]:
# 5. Prompt + Chain

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Pipeline LCEL: recupera → formata → prompt → llm → texto
chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Chain pronta!")

Chain pronta!


In [24]:
# 6. Loop de perguntas
print("Chat com PDF iniciado! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)

Chat com PDF iniciado! Digite 'sair' para encerrar.

Pergunta: olá


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Resposta: Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
e impactar positivamente os resultados da empresa, esta especialização é para você. Garanta sua vaga e se prepare para um futuro
promissor na gestão de pessoas e psicologia organizacional!Público-AlvoEste curso destina-se a profissionais que já possuam um

abordagem inovadora em Tech Recruiting, que utiliza tecnologias avançadas para atrair e reter os melhores talentos do mercado.
Além disso, o curso oferece uma formação prática que prepara você para lidar com as mudanças no ambiente de trabalho, promover

práticas e ferramentas para a resolução eficaz de conflitos e a gestão de erros no contexto de gestão de pessoas.
E-mail:
pos.toledo@pucpr.br
Telefone:
45991549135 www.pucpr.br

Pergunta: olá

Resposta:
--------------------------------------------------

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Resposta: Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
abordagem inovadora em Tech Recruiting, que utiliza tecnologias avançadas para atrair e reter os melhores talentos do mercado.
Além disso, o curso oferece uma formação prática que prepara você para lidar com as mudanças no ambiente de trabalho, promover

Legislação Trabalhista e Gestão de Pessoas
Esta disciplina tem como objetivo fornecer aos estudantes uma compreensão aprofundada das leis e regulamentações
trabalhistas relevantes no contexto brasileiro e como essas leis afetam a gestão de pessoas nas organizações. Os

práticas e ferramentas para a resolução eficaz de conflitos e a gestão de erros no contexto de gestão de pessoas.
E-mail:
pos.toledo@pucpr.br
Telefone:
45991549135 www.pucpr.br

Pergunta: O que este PDF trata

Resposta:
--------------------

KeyboardInterrupt: Interrupted by user

---
## Groq (gratuito)

1. Crie sua chave gratuita em: https://console.groq.com
2. E teste

In [26]:
# Testando o GROK
!pip install -q langchain-groq

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Cole aqui sua chave Groq
os.environ["GROQ_API_KEY"] = ""

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_groq
    | StrOutputParser()
)

print("Chain com Groq pronta! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)

Chain com Groq pronta! Digite 'sair' para encerrar.

Pergunta: O que trata este pdf

Resposta: Este documento parece tratar de um curso ou disciplina relacionada a gestão de pessoas e legislação trabalhista, com foco em práticas inovadoras de recrutamento e gestão de talentos, além de fornecer conhecimentos sobre leis e regulamentações trabalhistas no contexto brasileiro.
------------------------------------------------------------
Pergunta: Qual é o contexto deste pdf

Resposta: O contexto deste documento é a gestão de pessoas, especificamente práticas e ferramentas para a resolução eficaz de conflitos e a gestão de erros.
------------------------------------------------------------


KeyboardInterrupt: Interrupted by user